# 1. Import Libraries, Configuration Setup, and Load the Dataset

## 1.1 Install \& Import Libraries

In [ ]:
# !pip install peft==0.13.2, transformers==4.41.1

In [ ]:
# !pip install wordcloud

In [ ]:
# !pip install imblearn

In [ ]:
# !pip install lightgbm

In [ ]:
# ── Standard library ─────────────────────────────────────
import os
import re
import gc
import json
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, Tuple, Optional, List

# ── Core DS stack ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ── Scikit-learn ─────────────────────────────────────────
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    f1_score, roc_curve, auc, accuracy_score,
)
from sklearn.utils import resample, shuffle
from sklearn.utils.class_weight import compute_class_weight

# ── PyTorch ──────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Transformers ─────────────────────────────────────────
from transformers import (
    AutoTokenizer, AutoModel,
    AlbertModel, AlbertTokenizer,
    BertTokenizer,
)

# ── Notebook setup ───────────────────────────────────────
warnings.filterwarnings("ignore")


## 1.2 Seed \& Device

In [ ]:
SEED = 2025

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1.3 Configuration

In [ ]:
DRIVE_PROJECT_PATH = "./06_AspectLabel"
RAW_DATA_FILE = "data/mental_health_unified_labels_final.csv"
OUTPUT_DIR = "outputs"

ALBERT_MODEL_NAME = "albert-base-v2"
BIOBERT_MODEL_NAME = "dmis-lab/biobert-v1.1"

TEXT_COLUMN = "statement"

# ── Aspect labels (3-class each: NONE / WEAK / CLEAR) ───
ASPECT_NAMES = [
    "u_depression_strength", "u_anxiety_strength", "u_suicidal_strength",
    "u_stress_strength", "u_bipolar_strength", "u_personality_disorder_strength",
]
ASPECT_CLASSES = ["NONE", "WEAK", "CLEAR"]   # ordinal: 0 < 1 < 2
NUM_ASPECT_CLASSES = len(ASPECT_CLASSES)
ASPECT_LABEL_MAP = {v: i for i, v in enumerate(ASPECT_CLASSES)}

# ── Training config ──────────────────────────────────────
MAX_TOKEN_LENGTH = 200
BATCH_SIZE = 128                   # A100: large batches to saturate GPU
N_ITERATIONS = 10
RANDOM_STATE = 42
PATIENCE = 3
NUM_WORKERS = 4                    # A100 systems have more CPU cores

# ── A100 optimizations ──────────────────────────────────
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
USE_COMPILE = hasattr(torch, "compile")  # PyTorch 2.0+
print(f"BF16: {USE_BF16} | torch.compile: {USE_COMPILE}")

# ── Loss weights (λ) — equal weighting across conditions ─
LOSS_WEIGHTS = {
    "u_depression_strength": 1.0,
    "u_anxiety_strength": 1.0,
    "u_suicidal_strength": 1.0,
    "u_stress_strength": 1.0,
    "u_bipolar_strength": 1.0,
    "u_personality_disorder_strength": 1.0,
}

# ── Retrain flag ─────────────────────────────────────────
RETRAIN = True

# ── Derived paths ────────────────────────────────────────
RAW_DATA_PATH = os.path.join(DRIVE_PROJECT_PATH, RAW_DATA_FILE)
OUTPUT_PATH = Path(DRIVE_PROJECT_PATH) / OUTPUT_DIR
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "albert": {
        "model":  OUTPUT_PATH / "best_albert_aspect.pth",
        "params": OUTPUT_PATH / "best_albert_aspect_params.json",
    },
    "biobert": {
        "model":  OUTPUT_PATH / "best_biobert_aspect.pth",
        "params": OUTPUT_PATH / "best_biobert_aspect_params.json",
    },
}

print("--- Aspect-Only Multi-Task Configuration (Mental Health) ---")
print(f"  Project    : {DRIVE_PROJECT_PATH}")
print(f"  Aspects    : {ASPECT_NAMES} × {NUM_ASPECT_CLASSES} classes")
print(f"  Classes    : {ASPECT_CLASSES}")
print(f"  Loss wts   : {LOSS_WEIGHTS}")
print(f"  Batch size : {BATCH_SIZE}")

## 1.4 Mount drive & Load the data

In [ ]:
import os
if os.path.ismount("/content/drive"):
    # Already mounted, skip
    pass
else:
    !fusermount -u /content/drive 2>/dev/null
    !rm -rf /content/drive/*
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)

df_raw = pd.read_csv(RAW_DATA_PATH, index_col=0)
print(f"Loaded {df_raw.shape[0]} rows.")

df_raw.head()

In [ ]:
# ── SCOPE GATE: keep only evaluable posts ────────────────
SPECIAL_LABELS = {"OUT_OF_SCOPE", "INSUFFICIENT"}
before = len(df_raw)
df_raw = df_raw[~df_raw["u_label"].isin(SPECIAL_LABELS)].reset_index(drop=True)
print(f"Scope gating: {before} → {len(df_raw)} rows "
      f"(removed {before - len(df_raw)} OUT_OF_SCOPE/INSUFFICIENT)")

df_raw.head()

In [ ]:
df_raw = shuffle(df_raw, random_state=SEED).reset_index(drop=True)

# 2. Data Exploration \& Master Data Preparation

## 2.1 Master Data Preparation

### 2.1.1 Text cleaning

In [ ]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print("Cleaning text...")
df_clean = df_raw.copy()
df_clean[TEXT_COLUMN] = df_clean[TEXT_COLUMN].apply(clean_text)

### 2.1.2 Validate & encode aspect labels

In [ ]:
print("\nCondition strength label distributions:")
for asp in ASPECT_NAMES:
    if asp not in df_clean.columns:
        raise ValueError(f"Missing column: {asp}")

    # Coerce to uppercase string for consistent matching
    df_clean[asp] = df_clean[asp].astype(str).str.upper().str.strip()

    # Validate against ASPECT_CLASSES (NONE / WEAK / CLEAR)
    valid_mask = df_clean[asp].isin(ASPECT_CLASSES)
    n_invalid = (~valid_mask).sum()
    if n_invalid > 0:
        print(f"  WARNING: {asp} has {n_invalid} invalid values — setting to NONE")
        df_clean.loc[~valid_mask, asp] = "NONE"

    # Encode to integer using ASPECT_LABEL_MAP
    df_clean[f"{asp}_idx"] = df_clean[asp].map(ASPECT_LABEL_MAP)
    print(f"  {asp}: {df_clean[asp].value_counts().to_dict()}")

## 2.3 Master Data Split

In [ ]:
# Use first condition as stratification proxy
y_for_split = df_clean[f"{ASPECT_NAMES[0]}_idx"]
groups = df_clean[TEXT_COLUMN]

# 80/20 → train+val / test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(gss_test.split(df_clean, y_for_split, groups=groups))

split = np.full(len(df_clean), "train", dtype=object)
split[test_idx] = "test"

# train → train + val (75/25 of 80% = 60/20)
df_trainval = df_clean.iloc[trainval_idx].copy().reset_index(drop=True)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx2, val_idx = next(
    gss_val.split(df_trainval, df_trainval[f"{ASPECT_NAMES[0]}_idx"],
                  groups=df_trainval[TEXT_COLUMN])
)
split[trainval_idx[val_idx]] = "val"
df_clean["split"] = split

print("\nSplit counts:")
print(df_clean["split"].value_counts())

# Leakage check
group_split_nunique = df_clean.groupby(TEXT_COLUMN)["split"].nunique()
n_leaky = (group_split_nunique > 1).sum()
assert n_leaky == 0, f"{n_leaky} text groups leak across splits!"
print("Leakage check passed.")

# Save
split_path = OUTPUT_PATH / "master_split_multitarget.csv"
df_clean.to_csv(split_path, index=False)
print(f"Saved → {split_path}")

# 3. ALBERT \& BioBERT Models (Multi-aspect)

## 3.1 Shared helper functions

In [ ]:
def bootstrap_f1_ci(y_true, y_pred, n_iterations=1000, average="weighted"):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = []
    for _ in range(n_iterations):
        idx = resample(np.arange(len(y_true)))
        try:
            scores.append(f1_score(y_true[idx], y_pred[idx], average=average))
        except ValueError:
            continue
    if not scores:
        return np.nan, np.nan, np.nan
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


def bootstrap_auc_ci(y_true, y_scores, n_iterations=1000, average="macro"):
    y_true, y_scores = np.asarray(y_true), np.asarray(y_scores)
    scores = []
    for _ in range(n_iterations):
        idx = resample(np.arange(len(y_true)))
        try:
            scores.append(
                roc_auc_score(y_true[idx], y_scores[idx],
                              average=average, multi_class="ovr")
            )
        except ValueError:
            continue
    if not scores:
        return np.nan, np.nan, np.nan
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


def plot_confusion_matrix(y_true, y_pred, class_names, title="Model", save_dir=None):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix — {title}")
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f"{title}_confusion_matrix.png")
        plt.savefig(path, dpi=300)
        print(f"Saved → {path}")
    plt.show()


## 3.2 Data splits \& labels

In [ ]:
df_train = df_clean[df_clean["split"] == "train"].copy()
df_val   = df_clean[df_clean["split"] == "val"].copy()
df_test  = df_clean[df_clean["split"] == "test"].copy()

train_texts = df_train[TEXT_COLUMN].astype(str).tolist()
val_texts   = df_val[TEXT_COLUMN].astype(str).tolist()
test_texts  = df_test[TEXT_COLUMN].astype(str).tolist()

# Aspect labels (hard, 3-class each: NONE / WEAK / CLEAR)
y_train_aspects = {asp: df_train[f"{asp}_idx"].to_numpy(dtype=np.int64) for asp in ASPECT_NAMES}
y_val_aspects   = {asp: df_val[f"{asp}_idx"].to_numpy(dtype=np.int64) for asp in ASPECT_NAMES}
y_test_aspects  = {asp: df_test[f"{asp}_idx"].to_numpy(dtype=np.int64) for asp in ASPECT_NAMES}

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
for asp in ASPECT_NAMES:
    print(f"{asp}: {np.bincount(y_train_aspects[asp], minlength=NUM_ASPECT_CLASSES)}")

## 3.3 Generic multi-task model componets

In [ ]:
# ── Pre-tokenization ─────────────────────────────────────
def batch_tokenize(texts, tokenizer, max_len):
    enc = tokenizer(
        [str(t) for t in texts],
        truncation=True, padding="max_length",
        max_length=max_len, return_tensors="pt",
    )
    return enc["input_ids"], enc["attention_mask"]


# ── Aspect-Only Dataset ─────────────────────────────────
class AspectDataset(Dataset):
    """Pre-tokenized dataset with aspect labels only."""

    def __init__(self, input_ids, attention_masks, aspect_labels_dict):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.aspect_labels = {
            asp: np.asarray(labels, dtype=np.int64)
            for asp, labels in aspect_labels_dict.items()
        }

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
        }
        for asp in ASPECT_NAMES:
            item[f"{asp}_label"] = torch.tensor(
                self.aspect_labels[asp][idx], dtype=torch.long
            )
        return item


# ── Helper: short display name for an aspect column ─────
def asp_display(asp):
    """u_depression_strength → depression"""
    return asp.replace("u_", "").replace("_strength", "")


# ── Aspect-Only Classifier ──────────────────────────────
class AspectTransformer(nn.Module):
    """Shared transformer backbone with 6 aspect heads (3-class each).
    Each head predicts NONE / WEAK / CLEAR for one condition."""

    def __init__(self, base_model, hidden_size,
                 num_aspect_classes=NUM_ASPECT_CLASSES,
                 dropout=0.1):
        super().__init__()
        self.base = base_model
        self.dropout = nn.Dropout(dropout)
        self.aspect_heads = nn.ModuleDict({
            asp: nn.Linear(hidden_size, num_aspect_classes)
            for asp in ASPECT_NAMES
        })

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return {asp: head(cls) for asp, head in self.aspect_heads.items()}


# ── Class weights for each task ──────────────────────────
def compute_task_weights(labels, num_classes, device=DEVICE):
    """Compute balanced class weights for a single task."""
    unique = np.unique(labels)
    cw = compute_class_weight("balanced", classes=unique, y=labels)
    full_cw = np.ones(num_classes, dtype=np.float32)
    for cls_id, w in zip(unique, cw):
        full_cw[cls_id] = w
    return torch.tensor(full_cw, dtype=torch.float32).to(device)


def train_aspect_model(model, train_loader, val_loader,
                       aspect_weights_dict,
                       lr=2e-5, max_epochs=8, patience=PATIENCE):
    """Aspect-only multi-task training with equal weights.
    BF16 autocast for A100. Best epoch by mean aspect weighted F1."""
    model.to(DEVICE)

    # torch.compile for A100 (PyTorch 2.0+)
    if USE_COMPILE:
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # CE losses with class weights
    aspect_criteria = {
        asp: nn.CrossEntropyLoss(weight=aspect_weights_dict[asp])
        for asp in ASPECT_NAMES
    }

    best_f1, best_state, best_epoch = -1.0, None, -1
    no_improve = 0

    for epoch in range(1, max_epochs + 1):
        # ── Train ────────────────────────────────────────
        model.train()
        epoch_losses = []
        for batch in train_loader:
            ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad()

            # BF16 autocast for A100
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                aspect_logits = model(ids, mask)

                loss = torch.tensor(0.0, device=DEVICE)
                for asp in ASPECT_NAMES:
                    asp_y = batch[f"{asp}_label"].to(DEVICE, non_blocking=True)
                    loss = loss + LOSS_WEIGHTS[asp] * aspect_criteria[asp](
                        aspect_logits[asp], asp_y
                    )

            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        train_loss = float(np.mean(epoch_losses))

        # ── Validate ─────────────────────────────────────
        model.eval()
        val_asp_preds = {asp: [] for asp in ASPECT_NAMES}
        val_asp_true  = {asp: [] for asp in ASPECT_NAMES}

        with torch.no_grad():
            for batch in val_loader:
                ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
                mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                    aspect_logits = model(ids, mask)

                for asp in ASPECT_NAMES:
                    val_asp_preds[asp].extend(
                        aspect_logits[asp].float().argmax(dim=1).cpu().numpy()
                    )
                    val_asp_true[asp].extend(batch[f"{asp}_label"].numpy())

        # Mean aspect weighted F1 as selection criterion
        asp_f1s = {}
        for asp in ASPECT_NAMES:
            asp_f1s[asp] = f1_score(val_asp_true[asp], val_asp_preds[asp],
                                    average="weighted")
        mean_f1 = np.mean(list(asp_f1s.values()))

        asp_str = " | ".join(f"{asp_display(a)}={v:.3f}" for a, v in asp_f1s.items())
        print(f"  Epoch {epoch}/{max_epochs} | Loss={train_loss:.4f} | "
              f"MeanF1={mean_f1:.4f} | {asp_str}")

        # Best by mean aspect F1
        # Handle compiled model: extract _orig_mod if compiled
        raw = model._orig_mod if hasattr(model, "_orig_mod") else model
        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_state = {k: v.cpu().clone() for k, v in raw.state_dict().items()}
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    if best_state:
        raw = model._orig_mod if hasattr(model, "_orig_mod") else model
        raw.load_state_dict(best_state)
    return best_f1, raw, best_epoch


# =========================================================
# EVALUATION
# =========================================================
def evaluate_aspects(model, test_loader, model_name, save_dir=None):
    """Evaluate all aspect heads with bootstrap CIs."""
    model.eval()
    all_asp_logits = {asp: [] for asp in ASPECT_NAMES}
    all_asp_true   = {asp: [] for asp in ASPECT_NAMES}

    with torch.no_grad():
        for batch in test_loader:
            ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                aspect_logits = model(ids, mask)

            for asp in ASPECT_NAMES:
                all_asp_logits[asp].append(aspect_logits[asp].float().cpu().numpy())
                all_asp_true[asp].extend(batch[f"{asp}_label"].numpy())

    results = {}
    for asp in ASPECT_NAMES:
        asp_logits_np = np.concatenate(all_asp_logits[asp])
        asp_probs = torch.softmax(torch.tensor(asp_logits_np), dim=1).numpy()
        asp_preds = asp_probs.argmax(axis=1)
        asp_true = np.asarray(all_asp_true[asp])

        short = asp_display(asp)
        print(f"\n{'='*60}")
        print(f"{model_name} — {short.upper()} (3-class: NONE/WEAK/CLEAR)")
        print(f"{'='*60}")
        print(classification_report(
            asp_true, asp_preds,
            labels=list(range(NUM_ASPECT_CLASSES)),
            target_names=ASPECT_CLASSES, digits=4, zero_division=0,
        ))

        f1_m, f1_lo, f1_hi = bootstrap_f1_ci(asp_true, asp_preds)
        print(f"Weighted F1: {f1_m:.4f}  95% CI [{f1_lo:.4f}, {f1_hi:.4f}]")

        plot_confusion_matrix(asp_true, asp_preds, ASPECT_CLASSES,
                              title=f"{model_name}_Aspect_{short}",
                              save_dir=save_dir)

        results[asp] = {
            "y_true": asp_true, "y_pred": asp_preds, "y_prob": asp_probs,
            "f1_weighted": f1_m, "f1_ci": (f1_lo, f1_hi),
        }

    return results


def build_loaders(tokenizer, model_label):
    """Tokenize and build train/val/test loaders."""
    print(f"\nTokenizing for {model_label}...")
    train_ids, train_masks = batch_tokenize(train_texts, tokenizer, MAX_TOKEN_LENGTH)
    val_ids, val_masks     = batch_tokenize(val_texts, tokenizer, MAX_TOKEN_LENGTH)
    test_ids, test_masks   = batch_tokenize(test_texts, tokenizer, MAX_TOKEN_LENGTH)

    loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=True, persistent_workers=True)

    train_loader = DataLoader(
        AspectDataset(train_ids, train_masks, y_train_aspects),
        shuffle=True, **loader_kw)
    val_loader = DataLoader(
        AspectDataset(val_ids, val_masks, y_val_aspects),
        shuffle=False, **loader_kw)
    test_loader = DataLoader(
        AspectDataset(test_ids, test_masks, y_test_aspects),
        shuffle=False, **loader_kw)

    return train_loader, val_loader, test_loader


# ── Compute class weights ────────────────────────────────
aspect_weights = {
    asp: compute_task_weights(y_train_aspects[asp], NUM_ASPECT_CLASSES)
    for asp in ASPECT_NAMES
}
for asp in ASPECT_NAMES:
    print(f"{asp_display(asp)} class weights: {aspect_weights[asp].cpu().numpy().round(3)}")

## 3.4 ALBERT: Soft-label training

In [ ]:
print("\n" + "=" * 60)
print("ALBERT — Multi-Task Training")
print("=" * 60)

### 3.4.1 Pre-tokenize

In [ ]:
albert_tokenizer = AlbertTokenizer.from_pretrained(ALBERT_MODEL_NAME)
train_loader_albert, val_loader_albert, test_loader_albert = \
    build_loaders(albert_tokenizer, "ALBERT")

### 3.4.2 Hyperparameter tuning

In [ ]:
if RETRAIN:
    set_seed()
    start = time.time()

    param_space = {
        "lr":      (np.log10(1e-5), np.log10(3e-5)),
        "epochs":  (6, 10),
        "dropout": (0.0, 0.25),
    }

    best_f1_albert, best_params_albert, best_albert = 0, None, None

    for i in tqdm(range(N_ITERATIONS), desc="ALBERT Aspect Search"):
        hp = {
            "lr":      float(10 ** np.random.uniform(*param_space["lr"])),
            "epochs":  int(np.random.randint(*param_space["epochs"])),
            "dropout": float(np.random.uniform(*param_space["dropout"])),
        }

        base = AlbertModel.from_pretrained(ALBERT_MODEL_NAME)
        model = AspectTransformer(
            base, base.config.hidden_size, dropout=hp["dropout"],
        )

        f1_val, model, best_ep = train_aspect_model(
            model, train_loader_albert, val_loader_albert,
            aspect_weights,
            lr=hp["lr"], max_epochs=hp["epochs"],
        )

        if f1_val > best_f1_albert:
            best_f1_albert = f1_val
            best_params_albert = hp
            best_albert = model
            torch.save(model.state_dict(), MODEL_PATHS["albert"]["model"])

        gc.collect()
        torch.cuda.empty_cache()

    print(f"ALBERT tuning: {time.time()-start:.1f}s | Best F1={best_f1_albert:.4f}")
    print(f"Best params: {best_params_albert}")

    with open(MODEL_PATHS["albert"]["params"], "w") as f:
        json.dump(best_params_albert, f, indent=2)

else:
    with open(MODEL_PATHS["albert"]["params"]) as f:
        best_params_albert = json.load(f)
    base = AlbertModel.from_pretrained(ALBERT_MODEL_NAME)
    best_albert = AspectTransformer(
        base, base.config.hidden_size,
        dropout=best_params_albert.get("dropout", 0.1),
    )
    best_albert.load_state_dict(
        torch.load(MODEL_PATHS["albert"]["model"], map_location=DEVICE)
    )

best_albert.to(DEVICE)
best_albert.eval()

### 3.4.3 Model evaluation

In [ ]:
albert_results = evaluate_aspects(
    best_albert, test_loader_albert, "ALBERT", save_dir=str(OUTPUT_PATH)
)

## 3.5 BioBERT: Soft-label training

In [ ]:
print("\n" + "=" * 60)
print("BioBERT — Multi-Task Training")
print("=" * 60)

### 3.5.1 Pre-tokenize

In [ ]:
biobert_tokenizer = BertTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
train_loader_biobert, val_loader_biobert, test_loader_biobert = \
    build_loaders(biobert_tokenizer, "BioBERT")

### 3.5.2 Hyperparameter tuning

In [ ]:
if RETRAIN:
    set_seed()
    start = time.time()

    best_f1_biobert, best_params_biobert, best_biobert = 0, None, None

    for i in tqdm(range(N_ITERATIONS), desc="BioBERT Aspect Search"):
        hp = {
            "lr":      float(10 ** np.random.uniform(*param_space["lr"])),
            "epochs":  int(np.random.randint(*param_space["epochs"])),
            "dropout": float(np.random.uniform(*param_space["dropout"])),
        }

        base = AutoModel.from_pretrained(BIOBERT_MODEL_NAME)
        model = AspectTransformer(
            base, base.config.hidden_size, dropout=hp["dropout"],
        )

        f1_val, model, best_ep = train_aspect_model(
            model, train_loader_biobert, val_loader_biobert,
            aspect_weights,
            lr=hp["lr"], max_epochs=hp["epochs"],
        )

        if f1_val > best_f1_biobert:
            best_f1_biobert = f1_val
            best_params_biobert = hp
            best_biobert = model
            torch.save(model.state_dict(), MODEL_PATHS["biobert"]["model"])

        gc.collect()
        torch.cuda.empty_cache()

    print(f"BioBERT tuning: {time.time()-start:.1f}s | Best F1={best_f1_biobert:.4f}")
    print(f"Best params: {best_params_biobert}")

    with open(MODEL_PATHS["biobert"]["params"], "w") as f:
        json.dump(best_params_biobert, f, indent=2)

else:
    with open(MODEL_PATHS["biobert"]["params"]) as f:
        best_params_biobert = json.load(f)
    base = AutoModel.from_pretrained(BIOBERT_MODEL_NAME)
    best_biobert = AspectTransformer(
        base, base.config.hidden_size,
        dropout=best_params_biobert.get("dropout", 0.1),
    )
    best_biobert.load_state_dict(
        torch.load(MODEL_PATHS["biobert"]["model"], map_location=DEVICE)
    )

best_biobert.to(DEVICE)
best_biobert.eval()

### 3.5.3 Model Evaluation

In [ ]:
biobert_results = evaluate_aspects(
    best_biobert, test_loader_biobert, "BioBERT", save_dir=str(OUTPUT_PATH)
)

## 3.6 Save Prediction

In [ ]:
df_test_out = df_test[[TEXT_COLUMN]].copy()

# Aspect predictions
for asp in ASPECT_NAMES:
    short = asp_display(asp)
    df_test_out[f"albert_{short}_pred"] = albert_results[asp]["y_pred"]
    df_test_out[f"biobert_{short}_pred"] = biobert_results[asp]["y_pred"]
    df_test_out[f"{short}_true"] = albert_results[asp]["y_true"]

pred_path = OUTPUT_PATH / "test_predictions_aspect.csv"
df_test_out.to_csv(pred_path, index=False)
print(f"\nSaved predictions → {pred_path}")

# ── Summary ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY — Aspect-Only 3-Class (Mental Health)")
print("=" * 60)
for model_name, results in [("ALBERT", albert_results), ("BioBERT", biobert_results)]:
    print(f"\n{model_name}:")
    f1s = []
    for asp in ASPECT_NAMES:
        r = results[asp]
        short = asp_display(asp)
        print(f"  {short:25s} F1: {r['f1_weighted']:.4f} "
              f"[{r['f1_ci'][0]:.4f}, {r['f1_ci'][1]:.4f}]")
        f1s.append(r['f1_weighted'])
    print(f"  {'MEAN':25s} F1: {np.mean(f1s):.4f}")